# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [5]:
print("""
Finding 1:
The paper found that growing pages were younger than declining pages
(185 days vs 228 days), while their average word counts were almost the same.
My question is whether age is really the main signal here, or whether other
differences between the groups could also explain the result.

Finding 2:
The 5K+ word-count group had the highest average impressions and query
coverage, but the pattern was not linear across all word-count groups.
My question is whether the bucket comparison is enough to support the claim,
or whether differences in content mix and page type could also explain it.
""")


Finding 1:
The paper found that growing pages were younger than declining pages
(185 days vs 228 days), while their average word counts were almost the same.
My question is whether age is really the main signal here, or whether other
differences between the groups could also explain the result.

Finding 2:
The 5K+ word-count group had the highest average impressions and query
coverage, but the pattern was not linear across all word-count groups.
My question is whether the bucket comparison is enough to support the claim,
or whether differences in content mix and page type could also explain it.



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

data_path = "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count",
    "content_age_days",
    "days_since_last_update",
]

df_model = df.dropna(subset=features + ["client_id"]).copy()

# Week-5 model: The same number of clusters
k = 2

# Original result on the full data
scaler_full = StandardScaler()
X_full = scaler_full.fit_transform(df_model[features])

model_full = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_full = model_full.fit_predict(X_full)

before_score = silhouette_score(X_full, labels_full)

# Grouped split: clients do not appear in both sides
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

train_idx, test_idx = next(
    splitter.split(df_model, groups=df_model["client_id"])
)

train = df_model.iloc[train_idx]
test = df_model.iloc[test_idx]

scaler = StandardScaler()

X_train = scaler.fit_transform(train[features])
X_test = scaler.transform(test[features])

model = KMeans(n_clusters=k, random_state=42, n_init=10)

train_labels = model.fit_predict(X_train)
test_labels = model.predict(X_test)

after_score = silhouette_score(X_test, test_labels)

print(f"Before: full-data silhouette = {before_score:.3f}")
print(f"After: grouped holdout silhouette = {after_score:.3f}")
print(f"Train rows: {len(train)}")
print(f"Test rows: {len(test)}")
print(f"Train clients: {train['client_id'].nunique()}")
print(f"Test clients: {test['client_id'].nunique()}")

Before: full-data silhouette = 0.327
After: grouped holdout silhouette = 0.357
Train rows: 17210
Test rows: 4969
Train clients: 25
Test clients: 7


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
forbidden_features = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
]

used_forbidden = [col for col in features if col in forbidden_features]

duplicate_content = 0
if "content_id" in df_model.columns:
    duplicate_content = int(df_model["content_id"].duplicated().sum())

print("Features used:")
print(features)
print("\nLeakage check:")
print("Forbidden target/product fields:", used_forbidden)
print("Duplicate content IDs:", duplicate_content)
print("Future target window: not applicable — this is an unsupervised clustering model.")

Features used:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count', 'content_age_days', 'days_since_last_update']

Leakage check:
Forbidden target/product fields: []
Duplicate content IDs: 0
Future target window: not applicable — this is an unsupervised clustering model.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
print("""
The clustering grouped the pages into two performance patterns
using the features I selected.

I would use the groups to help organize the pages for review,
rather than treating them as fixed labels.

The results do not mean that one group causes better performance,
and the groups could change if the data or features change.
""")


The clustering grouped the pages into two performance patterns
using the features I selected.

I would use the groups to help organize the pages for review,
rather than treating them as fixed labels.

The results do not mean that one group causes better performance,
and the groups could change if the data or features change.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.